# Azure Blob Storage Cleanup Tool## Human-in-the-Loop Data Cleansing with LLM AssistanceThis notebook provides an end-to-end solution for cleaning Azure Blob Storage containers with:- **Conservative LLM pre-triage** that only auto-labels when completely certain (≥95% confidence)- **Human review UI** for edge cases and uncertain items  - **Iterative learning** through policy updates and few-shot examples- **Full audit trail** and safe deletion process### Quick Start Guide1. Copy `.env.example` to `.env` and fill in your credentials2. Run cells in order from top to bottom3. Start with a pilot run (set `MAX_FILES=100` in `.env`)4. Review items in the human review UI5. Run deletion only after 100% coverage and dry-run approval### Architecture- **Inventory**: Lists blobs and extracts text previews- **LLM Triage**: Background worker with 3 safety gates (confidence, self-consistency, policy alignment)- **Human Review**: Interactive UI for edge cases- **Deletion**: Safe execution with dry-run and audit logging

## 1. Setup and ConfigurationInstall dependencies and load environment variables.

In [ ]:
# Import the DataAtelier modulefrom dataatelier import AzureBlobCleanup, BlobCleanupConfigimport pandas as pdfrom IPython.display import display, HTMLimport os# Load environment variablesfrom dotenv import load_dotenvload_dotenv()print("✓ DataAtelier module loaded")print("✓ Environment variables loaded")

In [ ]:
# Verify configurationconfig = BlobCleanupConfig()print("Configuration:")print(f"  Container: {config.container_name}")print(f"  LLM Provider: {config.llm_provider}")  print(f"  Max Files: {'All' if config.max_files == -1 else config.max_files}")print(f"  Confidence Threshold: {config.confidence_threshold}")print(f"  Self-Consistency Runs: {config.self_consistency_runs}")if not config.storage_connection_string:    print("\n⚠️  Warning: AZURE_STORAGE_CONNECTION_STRING not set in .env")if not config.container_name:    print("⚠️  Warning: AZURE_STORAGE_CONTAINER_NAME not set in .env")

## 2. Create Blob InventoryList all blobs and extract text previews. This may take several minutes depending on container size.

In [ ]:
# Initialize the cleanup managercleanup = AzureBlobCleanup()# Create manifest (or load existing)if not os.path.exists('manifest.csv'):    print("Creating blob inventory...")    manifest_df = cleanup.inventory.create_manifest()    print(f"✓ Created manifest with {len(manifest_df)} blobs")else:    manifest_df = pd.read_csv('manifest.csv')    print(f"✓ Loaded existing manifest with {len(manifest_df)} blobs")# Show samplemanifest_df.head()

## 3. LLM Background TriageRun conservative LLM triage in batches. Only items passing ALL safety gates are auto-labeled.**Safety Gates:**1. **Confidence ≥95%** - High certainty required2. **Self-Consistency** - 3-5 independent runs must agree3. **Policy Alignment** - No NEVER DELETE violations

In [ ]:
# Run a batch of LLM triagebatch_size = 20  # Adjust based on API rate limitsresult = cleanup.run_triage_batch(batch_size=batch_size)print(f"✓ Processed {result['processed']} blobs")print(f"  Auto-labeled: {result['auto_labeled']}")print(f"    - Keep: {result['keep_count']}")print(f"    - Delete: {result['delete_count']}")print(f"  Needs review: {result['review_count']}")

## 4. Progress ReportCheck overall progress and queue status.

In [ ]:
# Show detailed progresscleanup.show_progress()

## 5. Human Review InterfaceReview items that need human judgment. Your decisions will:- Update the appropriate queue (keep/delete)- Add examples to few-shot learning- Improve LLM accuracy over time**Instructions:**- Read the file metadata and preview- Click **Keep** or **Delete**- Optionally add a reason to improve the LLM- Click **Skip** to review later

In [ ]:
# Create and display review UIreview_ui = cleanup.create_review_ui()display(review_ui)

## 6. Iterative ImprovementAfter human review sessions, re-run LLM triage. The LLM will learn from your decisions.

In [ ]:
# Re-run triage with updated examples and policyresult = cleanup.run_triage_batch(batch_size=20)print(f"✓ Re-triage complete")print(f"  Auto-labeled: {result['auto_labeled']}")print(f"  Needs review: {result['review_count']}")# Check progresscleanup.show_progress()

## 7. Update LLM Policy (Optional)Edit `llm_policy.md` to refine the decision rules. Then re-run triage batches.

In [ ]:
# View current policywith open('llm_policy.md', 'r') as f:    policy = f.read()    print("Current Policy (first 500 chars):")print(policy[:500])print("...")print(f"\nTotal length: {len(policy)} characters")print("\nTo edit: Open llm_policy.md in a text editor")

## 8. Deletion Process### Step 1: Dry Run (Review What Will Be Deleted)

In [ ]:
# Dry run - shows what would be deleted without actually deletingcleanup.deletion_manager.dry_run_deletion()

### Step 2: Verify 100% CoverageEnsure all blobs are labeled before deletion.

In [ ]:
# Check for unlabeled blobsunlabeled = cleanup.queue_manager.get_unlabeled_blobs()if len(unlabeled) > 0:    print(f"⚠️  Warning: {len(unlabeled)} blobs are still unlabeled")    print("   Complete labeling before deletion")    print("\nUnlabeled blobs:")    print(unlabeled[['name', 'size', 'content_type']].head(10))else:    print("✓ All blobs are labeled - ready for deletion")

### Step 3: Execute Deletion**⚠️  WARNING: This will permanently delete files (unless soft delete is enabled)**Only run this after:1. ✓ Dry run review completed2. ✓ 100% coverage verified  3. ✓ Deletion list exported and approved4. ✓ Soft delete enabled on storage account (recommended)

In [ ]:
# Export deletion list for final approvaldelete_df = pd.read_csv('to_delete.csv')delete_df.to_csv('deletion_approval.csv', index=False)print(f"✓ Deletion list exported to deletion_approval.csv ({len(delete_df)} files)")# EXECUTE DELETION (uncomment to run)# result = cleanup.deletion_manager.execute_deletion(dry_run=False)# print(f"✓ Deleted {result['deleted_count']} files")# print(f"  Failed: {result['failed_count']}")

## 9. Archive ArtifactsArchive all state files for future reference and compliance.

In [ ]:
# Archive artifactsarchive_dir = cleanup.archive_artifacts()print(f"✓ Artifacts archived to: {archive_dir}/")

## Appendix: Useful Commands### View Few-Shot Examples```pythonexamples = cleanup.llm_engine.load_few_shot_examples(max_examples=20)for ex in examples:    print(f"{ex['label']}: {ex['reason']}")```### View Audit Trail```pythonaudit_df = pd.read_csv('audit_log.csv')audit_df.tail(20)```### Re-process Specific Blob```pythonblob_name = "path/to/blob.txt"manifest = pd.read_csv('manifest.csv')blob_data = manifest[manifest['name'] == blob_name].iloc[0]decision = cleanup.llm_engine.triage_blob(blob_data)print(f"Label: {decision.label}, Confidence: {decision.confidence}")print(f"Reason: {decision.reason}")```### Export Queue Statistics```pythonimport pandas as pdstats = cleanup.queue_manager.get_statistics()pd.DataFrame([stats]).to_csv('queue_stats.csv', index=False)```